In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/default")
except Exception as ex:
    credential = InteractiveBrowserCredential()

In [ ]:
ml_client = MLClient.from_config(credential=credential)

In [ ]:
%%writefile src/sorvetes-training.py

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

print("Loading Data..")
sorvetes = pd.read_csv("vendas_sorvete.csv")

x, y = sorvetes[["Temperatura"]].values, sorvetes['Vendas de Sorvete'].values

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.20, random_state=0)


reg = 0.01

print("Training a logistic model with regularization rta of", reg)
model = LogisticRegression(C=1/reg, solver="liblinear").fit(x_train, y_train)

y_hat = model.predict(x_test)
acc = np.average(y_hat == y_test)
print("Accuracy: " + acc)

y_score = model.predict_proba(x_test)
auc = roc_auc_score(y_test, y_score[:,1])
print("AUC: " + str(auc))

In [ ]:
from azure.ai.ml import command

job = (
    code = "./src",
    command = "python sovertes-training.py",
    environment = "Azure_ML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute = "aml-cluster",
    display_name = "sovertes-pythonv2-train",
    experiments_name = "sovertes-training"
)

returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

FileNotFoundError: [Errno 2] No such file or directory: 'null/Users/arayarox'